# LitScope — Fine-Tuning Step 3: Train Behavioral Classifier

Trains a local Logistic Regression classifier on SPECTER embeddings to replace DeepSeek API calls.

**Pipeline:**
1. Load fine-tuned embeddings + `is_behavioral` labels
2. Train Logistic Regression
3. 5-fold cross-validation F1 evaluation
4. Agreement check against DeepSeek labels (50-paper sample)
5. Save classifier

**Prerequisite:** Run `04_finetune_specter.ipynb` first (need updated `papers_embeddings.npy`)

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_DIR  = '/content/drive/MyDrive/LitScope/data'
MODELS_DIR = '/content/drive/MyDrive/LitScope/models'

CLASSIFIED_CSV   = f'{DRIVE_DIR}/papers_classified.csv'
EMBEDDINGS_FILE  = f'{DRIVE_DIR}/papers_embeddings.npy'
CLASSIFIER_FILE  = f'{MODELS_DIR}/behavioral_classifier.pkl'

print(f'Classifier will be saved to: {CLASSIFIER_FILE}')

## 2. Install Dependencies

In [ ]:
!pip install scikit-learn pandas numpy joblib -q

## 3. Load Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(CLASSIFIED_CSV)
embeddings = np.load(EMBEDDINGS_FILE).astype('float32')

if len(df) != embeddings.shape[0]:
    raise ValueError(f'Row mismatch: CSV={len(df)}, embeddings={embeddings.shape[0]}')

# Only use papers with definitive labels (exclude uncertain)
mask = df['is_behavioral'].notna() & (df['confidence'] != 'uncertain')
X = embeddings[mask]
y = df[mask]['is_behavioral'].astype(int).values

print(f'Total papers:    {len(df)}')
print(f'Labeled (clean): {mask.sum()}')
print(f'Behavioral:      {y.sum()}  ({y.mean()*100:.1f}%)')
print(f'Non-behavioral:  {(1-y).sum()}  ({(1-y).mean()*100:.1f}%)')
print(f'Embedding shape: {X.shape}')

## 4. Train Logistic Regression

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import resample
import numpy as np
import joblib

# ── Label mapping: True/False → 'behavioral'/'other' ──────────────────────────
le = LabelEncoder()
y_str = np.where(y == 1, 'behavioral', 'other')
y_enc = le.fit_transform(y_str)
print(f'Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# ── Feature scaling ────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Oversample minority class (behavioral) to balance ─────────────────────────
beh_class = le.transform(['behavioral'])[0]
beh_mask  = (y_enc == beh_class)
n_majority = int((~beh_mask).sum())

X_beh_up, y_beh_up = resample(
    X_scaled[beh_mask], y_enc[beh_mask],
    n_samples=n_majority, random_state=42
)
X_balanced = np.vstack([X_scaled[~beh_mask], X_beh_up])
y_balanced = np.concatenate([y_enc[~beh_mask], y_beh_up])

# Shuffle
idx = np.random.RandomState(42).permutation(len(X_balanced))
X_balanced = X_balanced[idx]
y_balanced = y_balanced[idx]

print(f'Original:  behavioral={beh_mask.sum()}, other={(~beh_mask).sum()}')
print(f'Balanced:  behavioral={(y_balanced==beh_class).sum()}, other={(y_balanced!=beh_class).sum()}')

# ── Train MLP ─────────────────────────────────────────────────────────────────
clf = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128),
    activation='relu',
    solver='adam',
    alpha=0.001,
    batch_size=64,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=42,
    verbose=False,
)
clf.fit(X_balanced, y_balanced)

# ── Save scaler + classifier + label encoder together ─────────────────────────
model_bundle = {'scaler': scaler, 'clf': clf, 'le': le}
joblib.dump(model_bundle, CLASSIFIER_FILE)
print(f'\nClassifier trained and saved: {CLASSIFIER_FILE}')
print(f'Classes: {list(le.classes_)}  (multi-class ready)')

## 5. Evaluation — 5-Fold Cross-Validation

**Core test:** F1 score is the primary metric (balances precision and recall).
Target: F1 > 0.90. If below 0.85, consider MLP or additional features.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.utils import resample
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores_list   = []
prec_scores_list = []
rec_scores_list  = []

print('Running 5-fold cross-validation...')

for fold, (train_idx, val_idx) in enumerate(cv.split(X_scaled, y_enc)):
    X_tr, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_tr, y_val = y_enc[train_idx],    y_enc[val_idx]

    # Oversample behavioral in training fold only
    beh_mask_tr = (y_tr == beh_class)
    n_maj = int((~beh_mask_tr).sum())
    X_beh_up, y_beh_up = resample(
        X_tr[beh_mask_tr], y_tr[beh_mask_tr],
        n_samples=n_maj, random_state=42
    )
    X_tr_bal = np.vstack([X_tr[~beh_mask_tr], X_beh_up])
    y_tr_bal = np.concatenate([y_tr[~beh_mask_tr], y_beh_up])
    idx = np.random.RandomState(42).permutation(len(X_tr_bal))
    X_tr_bal, y_tr_bal = X_tr_bal[idx], y_tr_bal[idx]

    fold_clf = MLPClassifier(
        hidden_layer_sizes=(512, 256, 128),
        activation='relu',
        solver='adam',
        alpha=0.001,
        batch_size=64,
        max_iter=300,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=15,
        random_state=42,
        verbose=False,
    )
    fold_clf.fit(X_tr_bal, y_tr_bal)
    y_pred = fold_clf.predict(X_val)

    f1_scores_list.append(f1_score(y_val, y_pred, average='macro'))
    prec_scores_list.append(precision_score(y_val, y_pred, average='macro'))
    rec_scores_list.append(recall_score(y_val, y_pred, average='macro'))
    print(f'  Fold {fold+1}: F1={f1_scores_list[-1]:.3f}')

f1_scores   = np.array(f1_scores_list)
prec_scores = np.array(prec_scores_list)
rec_scores  = np.array(rec_scores_list)

print()
print('=' * 50)
print('  5-Fold Cross-Validation Results')
print('=' * 50)
print(f'  F1        per fold: {[f"{s:.3f}" for s in f1_scores]}')
print(f'  F1        mean ± std: {f1_scores.mean():.3f} ± {f1_scores.std():.3f}')
print()
print(f'  Precision mean ± std: {prec_scores.mean():.3f} ± {prec_scores.std():.3f}')
print(f'  Recall    mean ± std: {rec_scores.mean():.3f} ± {rec_scores.std():.3f}')
print()

if f1_scores.mean() >= 0.90:
    print('  ✅ F1 >= 0.90 — classifier meets quality threshold')
elif f1_scores.mean() >= 0.85:
    print('  ⚠️  F1 between 0.85–0.90 — acceptable, monitor in production')
else:
    print('  ❌ F1 < 0.85 — check embeddings or increase training data')
print('=' * 50)

## 6. Agreement Check — Classifier vs DeepSeek (50-paper sample)

Checks how often the trained classifier agrees with the original DeepSeek labels
on a random 50-paper sample. High agreement (>90%) means the classifier successfully
learned DeepSeek's classification logic.

In [ ]:
import random

labeled_df  = df[mask].copy().reset_index(drop=True)
labeled_emb = X_scaled  # use scaled embeddings

beh_idx    = labeled_df[labeled_df['is_behavioral'] == True].index.tolist()
nonbeh_idx = labeled_df[labeled_df['is_behavioral'] == False].index.tolist()

random.seed(42)
sample_idx = (
    random.sample(beh_idx, min(25, len(beh_idx))) +
    random.sample(nonbeh_idx, min(25, len(nonbeh_idx)))
)
sample_df  = labeled_df.iloc[sample_idx].reset_index(drop=True)
sample_emb = labeled_emb[sample_idx]

# Load saved bundle and predict
bundle     = joblib.load(CLASSIFIER_FILE)
clf_loaded = bundle['clf']
le_loaded  = bundle['le']

preds_enc  = clf_loaded.predict(sample_emb)
preds      = le_loaded.inverse_transform(preds_enc)        # 'behavioral' or 'other'
probs      = clf_loaded.predict_proba(sample_emb)
beh_class_idx = list(le_loaded.classes_).index('behavioral')
probs_beh  = probs[:, beh_class_idx]

# DeepSeek ground truth as string labels
deepseek_str = np.where(sample_df['is_behavioral'].astype(int) == 1, 'behavioral', 'other')
agreements   = (preds == deepseek_str)
agree_rate   = agreements.mean()

print('=' * 60)
print(f'  Agreement with DeepSeek labels  (n={len(sample_df)})')
print('=' * 60)
print(f'  Agreement rate: {agree_rate*100:.1f}%  ({agreements.sum()}/{len(agreements)})')
print()

disagree_mask = ~agreements
disagree_df   = sample_df[disagree_mask].copy()
disagree_preds = preds[disagree_mask]
disagree_probs = probs_beh[disagree_mask]

if len(disagree_df) == 0:
    print('  ✅ No disagreements in this sample')
else:
    print(f'  Disagreements ({len(disagree_df)} papers):')
    print(f'  {"DeepSeek":<12} {"Clf":<12} {"P(beh)":<8}  Title')
    print(f'  {"─"*10:<12} {"─"*10:<12} {"─"*6:<8}  {"─"*40}')
    for i, (_, row) in enumerate(disagree_df.iterrows()):
        ds  = 'behavioral' if row['is_behavioral'] else 'other'
        clf_pred = disagree_preds[i]
        print(f'  {ds:<12} {clf_pred:<12} {disagree_probs[i]:<8.3f}  {str(row["title"])[:55]}')
        print(f'  {"":<34} DeepSeek reason: {str(row.get("reason",""))[:55]}')

print()
if agree_rate >= 0.90:
    print('  ✅ >= 90% agreement — classifier ready for production')
elif agree_rate >= 0.80:
    print('  ⚠️  80–90% agreement — review disagreements above')
else:
    print('  ❌ < 80% agreement — check data quality or retrain')
print('=' * 60)

## 7. Summary

In [ ]:
print('=' * 60)
print('  Training Complete')
print('=' * 60)
print(f'  Classifier saved:   {CLASSIFIER_FILE}')
print(f'  Training samples:   {len(X)}')
print(f'  Classes:            {list(le.classes_)}')
print(f'  CV F1 (macro):      {f1_scores.mean():.3f} ± {f1_scores.std():.3f}')
print(f'  DeepSeek agreement: {agree_rate*100:.1f}%  (n=50 sample)')
print()
print('  Saved bundle contains: scaler + clf + label_encoder')
print()
print('  Next steps:')
print('  1. Copy behavioral_classifier.pkl to backend/data/')
print('  2. Update backend/litscope/classifier.py to load bundle')
print('  3. Copy updated papers_embeddings.npy to backend/data/')
print('=' * 60)